# 实验 16 — dbt snapshot（SCD Type 2）

**目标**：用 dbt snapshot 跟踪 `currencies_meta` 中维度属性的历史变化。

## 什么是 SCD2

维度表（dim_*）的属性会变（例如客户改名、商品分类调整）。SCD2 不覆盖旧值，而是给每个版本一个 `valid_from` / `valid_to`，留住历史。

dbt 的 snapshot 自动维护这两列 + 一个 `dbt_scd_id`（hash 主键），完美适配 SCD2。

## 本仓库的 snapshot

[dbt_project/snapshots/snp_currencies.sql](../dbt_project/snapshots/snp_currencies.sql)：
- `unique_key=currency_code`
- `strategy=check`，`check_cols=['currency_name']`：name 变了就插新版本
- 目标 schema：`snapshots`

In [ ]:
import subprocess, os, duckdb
DB='../data/sandbox.duckdb'

def dbt(*args):
    r = subprocess.run(['uv','run','dbt',*args], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ, 'DBT_PROFILES_DIR': '.'})
    return r.stdout + r.stderr

print(dbt('snapshot')[-1500:])

In [ ]:
con = duckdb.connect(DB)
print('=== snapshot 表结构 ===')
print(con.execute('describe snapshots.snp_currencies').df())
print()
print('=== 当前快照内容 ===')
con.execute("""
    select currency_code, currency_name, dbt_valid_from, dbt_valid_to
    from snapshots.snp_currencies
    order by currency_code, dbt_valid_from
""").df()

In [ ]:
# 模拟上游 dimension 发生变化：把 'US Dollar' 改成 'United States Dollar'
con.execute("update raw_ecb.currencies_meta set name='United States Dollar' where code='USD'")
print('upstream changed:', con.execute("select * from raw_ecb.currencies_meta where code='USD'").fetchall())

In [ ]:
# 再跑 snapshot，应该新增一行版本，并把旧版本的 dbt_valid_to 设为现在
print(dbt('snapshot')[-800:])
print()
con.execute("""
    select currency_code, currency_name, dbt_valid_from, dbt_valid_to
    from snapshots.snp_currencies
    where currency_code = 'USD'
    order by dbt_valid_from
""").df()

In [ ]:
# 把 USD 改回原状，便于其他实验复用
con.execute("update raw_ecb.currencies_meta set name='US Dollar' where code='USD'")
print(dbt('snapshot')[-300:])

## 生产环境踩坑提示

- **snapshot 必须定期跑**：每天 / 每小时 / 跟 ingest 同频。没人跑的 snapshot 等于没用，历史不在数据库里。
- **`strategy=check` vs `strategy=timestamp`**：
  - `timestamp` 需要上游表有 `updated_at` 列，dbt 比较时间戳判断变化（更快、更准）
  - `check` 比较选定列的值，适合上游没有审计列的情况（这里就是这种）
- **下游怎么用**：拼时间范围 join。`where snapshot.dbt_valid_from <= event_time AND (dbt_valid_to is null or event_time < dbt_valid_to)` 得到事件发生时的维度值。
- **不可逆**：snapshot 是 append-only。手工删一行很危险，下次跑会重新插。如果定义错了要重置，`dbt build --select snp_xxx --full-refresh` 也不行，必须手工 drop。
- **PII**：snapshot 留住的是历史值，意味着删 PII 时要联动 snapshot 表，否则违反 GDPR 等。

## 思考题

- `strategy=timestamp` 长什么样？如果上游有 `updated_at`，怎么改 snapshot 定义？
- 想统计“2024-06-01 那天 USD 叫什么名字”，怎么写 SQL？
- 把 snapshot 当作 source，让 marts 模型按时间 join，避免“最新维度”污染历史事实。